# BTL02 T0401 — RQ3 Data Augmentation Ablation (Kaggle GPU)

Bật GPU accelerator rồi chạy 7 code cells theo thứ tự. Runner chấp nhận T4 x2/P100 (và cảnh báo tên GPU khác), giữ nguyên PyTorch/CUDA Kaggle. Đầu ra tải được qua Save Version → Quick Save nằm trong `/kaggle/working/BTL02/T0401/`. Auto-Skip chỉ bỏ qua MLflow runs `FINISHED`; không tự resume một epoch bị ngắt. Secrets cần có: `GITHUB_TOKEN` (repository private) và `KAGGLE_API_TOKEN` (chỉ dùng nếu không thêm competition input).

In [ ]:
import os, platform, shutil, subprocess, sys, tempfile
from pathlib import Path

BRANCH = "exp/BTL02_T0401_ablation_augmentation"
REPO_URL = "https://github.com/Duy-DS/Deep_Learning_Group.git"
REPO_DIR = Path("/kaggle/working/Deep_Learning_Group")
PROJECT_DIR = REPO_DIR / "BTL02"
OUTPUT_ROOT = Path("/kaggle/working/BTL02/T0401")
MLRUNS_DIR = OUTPUT_ROOT / "mlruns"
CHECKPOINTS_DIR = OUTPUT_ROOT / "checkpoints"
LOGS_DIR = OUTPUT_ROOT / "logs"
CSV_PATH = OUTPUT_ROOT / "t0401_ablation_augmentation_24col.csv"
CSV_NAME = CSV_PATH.name
for folder in (OUTPUT_ROOT, MLRUNS_DIR, CHECKPOINTS_DIR, LOGS_DIR): folder.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
import torch
if not torch.cuda.is_available():
    raise RuntimeError("B?t GPU Accelerator trong Kaggle Settings tr??c khi hu?n luy?n.")
print("PyTorch:", torch.__version__)
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    name = props.name.upper()
    print(f"GPU {index}: {props.name} | {props.total_memory / 1024**3:.1f} GiB")
    if not any(kind in name for kind in ("T4", "P100")):
        print("C?NH B?O: GPU kh?ng ph?i T4/P100 ph? bi?n; ki?m tra VRAM tr??c khi ch?y.")
os.environ["MLFLOW_TRACKING_URI"] = MLRUNS_DIR.resolve().as_uri()
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
(MLRUNS_DIR / ".trash").mkdir(parents=True, exist_ok=True)
print("Kaggle output root:", OUTPUT_ROOT)


In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

def read_secret(name):
    try:
        value = secrets.get_secret(name)
        return value.strip() if value else None
    except Exception:
        return None

def clone_or_update_branch():
    token = read_secret("GITHUB_TOKEN")
    if token and token.startswith("x-access-token:"):
        token = token.split(":", 1)[1]
    git_env = os.environ.copy()
    askpass_path = None
    if token:
        fd, askpass_path = tempfile.mkstemp(prefix="t0401-git-askpass-")
        with os.fdopen(fd, "w", encoding="utf-8") as stream:
            stream.write('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$T0401_GITHUB_TOKEN" ;; esac\n')
        os.chmod(askpass_path, 0o700)
        git_env.update({"GIT_ASKPASS": askpass_path, "GIT_TERMINAL_PROMPT": "0", "T0401_GITHUB_TOKEN": token})
    try:
        if REPO_DIR.exists():
            subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True, env=git_env)
            subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
            subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
        else:
            subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True, env=git_env)
    finally:
        if askpass_path and os.path.exists(askpass_path): os.remove(askpass_path)
        git_env.pop("T0401_GITHUB_TOKEN", None)
        token = None
clone_or_update_branch()
os.chdir(PROJECT_DIR)
assert subprocess.check_output(["git", "branch", "--show-current"], text=True).strip() == BRANCH
print("Checked out project branch:", BRANCH)


In [ ]:
# Restore only complete MLflow runs from an attached read-only Kaggle Dataset backup.
input_root = Path("/kaggle/input/datasets")
backup_roots = []
if input_root.exists():
    for current, dirs, files in os.walk(input_root):
        current_path = Path(current)
        if current_path.name == "mlruns" and (current_path / "0" / "meta.yaml").is_file():
            backup_roots.append(current_path)
            dirs[:] = []

def restore_finished_runs(source_root, destination_root):
    """Ch?p experiment metadata v? t?ng run FINISHED c?n thi?u; kh?ng ghi ?? run local."""
    from mlflow.tracking import MlflowClient
    source_root, destination_root = Path(source_root), Path(destination_root)
    source_client = MlflowClient(tracking_uri=source_root.resolve().as_uri())
    restored = []
    for experiment in source_client.search_experiments():
        source_experiment = source_root / str(experiment.experiment_id)
        destination_experiment = destination_root / str(experiment.experiment_id)
        destination_experiment.mkdir(parents=True, exist_ok=True)
        source_meta = source_experiment / "meta.yaml"
        destination_meta = destination_experiment / "meta.yaml"
        if source_meta.is_file() and not destination_meta.exists():
            shutil.copy2(source_meta, destination_meta)
        runs = source_client.search_runs(
            [experiment.experiment_id],
            filter_string="attributes.status = 'FINISHED'",
            max_results=10000,
        )
        for run in runs:
            source_run = source_experiment / run.info.run_id
            destination_run = destination_experiment / run.info.run_id
            if not source_run.is_dir() or destination_run.exists():
                continue
            shutil.copytree(source_run, destination_run)
            restored.append(run.info.run_id)
    return restored

restored_run_ids = []
for backup_root in backup_roots:
    restored_run_ids.extend(restore_finished_runs(backup_root, MLRUNS_DIR))
(MLRUNS_DIR / ".trash").mkdir(parents=True, exist_ok=True)
os.environ["MLFLOW_TRACKING_URI"] = MLRUNS_DIR.resolve().as_uri()
print(f"Restored {len(restored_run_ids)} FINISHED backup runs; local existing runs were not overwritten.")


In [ ]:
raw_csv = PROJECT_DIR / "data/raw/fer2013.csv"
raw_csv.parent.mkdir(parents=True, exist_ok=True)
competition_root = Path("/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge")
archive = competition_root / "fer2013.tar.gz"

if not raw_csv.is_file() and archive.is_file():
    import tempfile
    from src.data.downloader import FER2013Downloader
    with tempfile.TemporaryDirectory(prefix="t0401-fer2013-", dir=OUTPUT_ROOT) as temporary:
        candidates = FER2013Downloader._candidates(archive, archive.name, Path(temporary))
        if not candidates:
            raise RuntimeError("Competition archive kh?ng ch?a fer2013.csv.")
        FER2013Downloader.validate_csv(candidates[0])
        shutil.copy2(candidates[0], raw_csv)

if not raw_csv.is_file():
    token = read_secret("KAGGLE_API_TOKEN")
    if not token:
        raise RuntimeError("Kh?ng t?m th?y competition input/CSV; th?m Kaggle Secret KAGGLE_API_TOKEN.")
    os.environ["KAGGLE_API_TOKEN"] = token
    token = None
    try:
        subprocess.run([sys.executable, "-m", "src.data.downloader"], cwd=PROJECT_DIR, check=True)
    finally:
        os.environ.pop("KAGGLE_API_TOKEN", None)

from src.data.downloader import FER2013Downloader
split_counts = FER2013Downloader.validate_csv(raw_csv)
assert split_counts == {"Training": 28709, "PublicTest": 3589, "PrivateTest": 3589}, split_counts
print("Verified FER2013 splits:", split_counts)

outputs = PROJECT_DIR / "outputs"
outputs.mkdir(exist_ok=True)
for name, target in (("checkpoints", CHECKPOINTS_DIR), ("logs", LOGS_DIR)):
    link = outputs / name
    if link.is_symlink():
        if link.resolve() != target.resolve():
            raise RuntimeError(f"{link} symlink tr? sai; d?ng an to?n.")
    elif link.exists():
        if any(item.name != ".gitkeep" for item in link.iterdir()):
            raise RuntimeError(f"{link} c? d? li?u; kh?ng ghi ??.")
        for item in link.iterdir():
            item.unlink()
        link.rmdir()
    if not link.exists():
        link.symlink_to(target, target_is_directory=True)


In [ ]:
import numpy as np
import pandas as pd
import yaml
from mlflow.tracking import MlflowClient
from src.train.mlflow_logger import MLflowLogger
from src.train.trainer import Trainer

CONFIG_FILES = ["exp09_aug_none.yaml", "exp09_aug_basic.yaml", "exp10_aug_advanced.yaml"]
raw_configs = []
for filename in CONFIG_FILES:
    config_path = Path("experiments/ablation") / filename
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    config["config_path"] = str(config_path)
    config["run_name"] = Trainer._format_run_name(
        exp_id=config["exp_id"], architecture=config["architecture"],
        optimizer=config["optimizer"].lower(), pretrained=config["pretrained"],
        image_size=config["image_size"], augmentation_mode=config["augmentation_mode"])
    raw_configs.append(config)

def sanitize_to_python_scalars(record):
    def convert(value):
        if isinstance(value, np.generic): return convert(value.item())
        if isinstance(value, np.ndarray): return convert(value.item()) if value.ndim == 0 else str(convert(value.tolist()))
        if isinstance(value, dict): return str({str(key): convert(item) for key, item in value.items()})
        if isinstance(value, (list, tuple)): return str([convert(item) for item in value])
        if value is None or type(value) in (bool, int, float, str): return value
        return str(value)
    return {str(key): convert(value) for key, value in record.items()}

manifest = pd.DataFrame([sanitize_to_python_scalars(config) for config in raw_configs])
client = MlflowClient(tracking_uri=os.environ["MLFLOW_TRACKING_URI"])
experiment = client.get_experiment_by_name("BTL02_FER2013_PyTorch")
finished_names = set()
if experiment:
    for run in client.search_runs([experiment.experiment_id], max_results=10000):
        name = run.data.tags.get("mlflow.runName")
        if run.info.status == "FINISHED" and name: finished_names.add(name)
manifest["status"] = ["DONE" if name in finished_names else "PENDING" for name in manifest["run_name"]]
display(manifest[["exp_id", "augmentation_mode", "seed", "batch_size", "epochs", "run_name", "status"]])


In [ ]:
sync_lock = threading.Lock()

def sync_and_export_results():
    with sync_lock:
        logger = MLflowLogger()
        raw_export = LOGS_DIR / "t0401_all_runs_export.csv"
        logger.export_summary_csv(str(raw_export))
        all_rows = pd.read_csv(raw_export, encoding="utf-8-sig", keep_default_na=False)
        assert all_rows.columns.tolist() == MLflowLogger.CSV_COLUMNS
        names = {item["run_name"] for item in raw_configs}
        selected = all_rows[all_rows["Name"].astype(str).isin(names)].copy()
        selected = selected[selected["Status"].astype(str).str.upper().eq("FINISHED")]
        safe = MLflowLogger.normalize_records(selected.to_dict(orient="records"))
        result = pd.DataFrame(safe, columns=MLflowLogger.CSV_COLUMNS)
        temporary = CSV_PATH.with_name(CSV_PATH.name + ".sync-tmp")
        result.to_csv(temporary, index=False, encoding="utf-8-sig")
        os.replace(temporary, CSV_PATH)
        raw_export.unlink(missing_ok=True)
        print(f"Exported {len(result)} FINISHED runs / 3 to {CSV_PATH}", flush=True)
        return result

print("MLflow:", MLRUNS_DIR)
print("Checkpoint output:", CHECKPOINTS_DIR)
print("CSV schema:", len(MLflowLogger.CSV_COLUMNS), "columns")


In [ ]:
failed_runs = []
for config in raw_configs:
    run_name = config["run_name"]
    experiment = client.get_experiment_by_name("BTL02_FER2013_PyTorch")
    matches = client.search_runs([experiment.experiment_id], filter_string=f"tags.mlflow.runName = '{run_name}'", max_results=100) if experiment else []
    if any(run.info.status == "FINISHED" for run in matches):
        print(f"[AUTO-SKIP] FINISHED: {run_name}", flush=True)
        sync_and_export_results()
        continue
    print(f"\n===== {config['exp_id']} / {config['augmentation_mode']} =====", flush=True)
    process = subprocess.Popen([sys.executable, "-u", "-m", "src.train.trainer", "--config", config["config_path"]],
                               cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env=os.environ.copy())
    for line in process.stdout: print(line, end="", flush=True)
    code = process.wait()
    if code:
        failed_runs.append({"run_name": run_name, "return_code": code})
        print(f"[FAILED] exit={code}; gi? artifacts hi?n c?, ti?p t?c run sau.", flush=True)
    else: print(f"[DONE] {run_name}", flush=True)
    sync_and_export_results()
final_results = sync_and_export_results()
if failed_runs: print("Runs c?n ki?m tra/ch?y l?i:", failed_runs)
display(final_results)
print("Save Version ? Quick Save ?? l?u to?n b?:", OUTPUT_ROOT)
